In [ ]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

import os
from getpass import getpass
import pandas as pd
import json
import re
from tqdm import tqdm
from sklearn.metrics import classification_report
from json.decoder import JSONDecodeError

import nest_asyncio
from dotenv import load_dotenv
from openai import OpenAI

pd.set_option('max_colwidth', 400)

nest_asyncio.apply()

load_dotenv('.env', verbose=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [ ]:
openrouter_api_key = os.getenv("OLLAMA")

BASEURL = "http://localhost:11434/v1"

In [ ]:
client = OpenAI(
    base_url=BASEURL,
    api_key=openrouter_api_key,
)
model_name = "qwen2.5:7b"

In [ ]:
def clean_json_response(response: str) -> str:
    """
    Убираем в ответе llm markdown (```json и ```)
    """
    response = response.strip()
    if response.startswith("```json"):
        response = response[7:]
    elif response.startswith("```"):
        response = response[3:]

    if response.endswith("```"):
        response = response[:-3]

    return response.strip()

In [ ]:
def get_completion(prompt: str, system_prompt="", prefill="", temperature=0):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    if prefill:
        messages.append({"role": "assistant", "content": prefill})

    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=1000,
        temperature=temperature,  # 0.7
        top_p=0.9,
    )
    return response.choices[0].message.content

In [31]:
# Тестируем:
get_completion("Привет!")

'Привет! Как я могу вам помочь сегодня?'

# Препроцессинг

In [ ]:
categories = [
    "Ролики и скейтбординг",
    "Настольные игры",
    "Теннис бадминтон пинг-понг",
    "Пейнтбол и страйкбол",
    "Единоборства",
    "Бильярд и боулинг",
    "Фитнес и тренажёры",
    "Туризм",
    "Игры с мячом",
    "Зимние виды спорта",
    "Дайвинг и водный спорт",
    "Спортивное питание",
]

In [ ]:
# test_df и val_df лежат на степике
test_df = pd.read_csv("test_df_llm_generation_1.csv")[["title", "item_id"]]
val_df = pd.read_csv("val_df_llm_generation.csv")
test_df.sample(10)

,title,item_id
23,Питьевая система (гидратор) camelbak ThermoBak 3л,23
46,Боевой тактический пояс варбелт Ссо воин,46
96,Аква-коврик,96
26,Поповский кий Баринова черный эбен,26
15,"Стойки универсальные с механизмом натяжения, в стаканах с крышками A15003",15
113,Спин-байк bronze GYM S900 PRO,113
72,Налобный фонарь Xiaomi NexTool NE20002 черный,72
104,Колонизаторы Catan/Катан Города и рыцари (дополн),104
102,Игра данетки,102
57,Бцаа,57


In [ ]:
def process_dataframe_with_cache(
    test_df, system_prompt, prefill, cache_file="answer_cache.json"
):
    key_to_answer = {}
    if os.path.exists(cache_file):
        with open(cache_file, "r", encoding="utf-8") as f:
            key_to_answer = json.load(f)
    answers = []

    for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
        title = row["title"]
        item_id = row["item_id"]
        key = f"{system_prompt}_{title}"

        if key in key_to_answer:
            answer = key_to_answer[key]
        else:
            prompt = f"<text>{title}</text>"
            answer = get_completion(
                prompt, system_prompt=system_prompt, prefill=prefill
            )
            key_to_answer[key] = answer
            # Сохраняем кэш после каждого нового ответа
            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(key_to_answer, f, ensure_ascii=False, indent=2)
        answers.append(answer)

    return answers

# Промты для моделей

In [ ]:
thinking_model_role_prompt = """Ты — ведущий эксперт по классификации товаров на маркетплейсе. 
Твоя задача: глубоко проанализировать название и определить истинное назначение товара."""

thinking_model_rule_prompt = """
КРИТИЧЕСКИЕ ПРАВИЛА АНАЛИЗА:
1. ТАКТИКА И ОРУЖИЕ: Если видишь бренды Gletcher, Remington, Strike, или слова "подсумок", "кобура", "пулемет", "лоадер", "бронежилет", "тактический" — это ВСЕГДА 'Пейнтбол и страйкбол'.
2. БИЛЬЯРД: Слова "шафт", "кий", "мел для кия", "пул", "пирамида" (в контексте столов) — это 'Бильярд и боулинг'.
3. ВОДА: "Сап", "SUP", "байдарка", "гидрокостюм", "ласты" — это 'Дайвинг и водный спорт'.
4. ЗИМА: "Лавинное", "сноуборд", "лыжи", "коньки", "фигурное катание" — это 'Зимние виды спорта'.
5. ПИТАНИЕ: "Омега", "Витамины", "Протеин", "Гейнер", "Ашваганда" — это 'Спортивное питание'."""
thinking_model_prefill = """Анализ: """
thinking_prompt = """
Алгоритм ответа:
- Что это за предмет?
- К какому специфичному виду спорта он относится?
- Итоговая категория из списка: """ + ", ".join(categories)

thinking_model_few_shot_prompt = """
Примеры правильного анализа:
Вход: <text>Gletcher cst 304</text>
Анализ: Gletcher — бренд пневматического и страйкбольного оружия. Модель CST 304 — это пистолет. Категория: Пейнтбол и страйкбол.
Вход: <text>Карбоновый шафт 12 мм (пул)</text>
Анализ: Шафт — это верхняя часть бильярдного кия. Пул — разновидность бильярда. Категория: Бильярд и боулинг.
Вход: <text>Омега</text>
Анализ: В спортивных товарах Омега — это жирные кислоты (БАД). Категория: Спортивное питание.
Вход: <text>Лавинный щуп</text>
Анализ: Это снаряжение для безопасности в горах при занятиях фрирайдом или лыжным спортом. Категория: Зимние виды спорта.
"""

role_prompt = "Ты — строгий классификатор. Ты получаешь название товара и его анализ. Твоя задача — выдать JSON с названием категории."

rules_prompt = (
    """
ПРАВИЛА:
1. Выбери ОДНУ категорию из списка: """
    + "\n".join([f"- {c}" for c in categories])
    + """
2. Твой ответ должен быть СТРОГО в формате JSON: {"category": "название"}
3. Не меняй названия категорий (никаких лишних знаков, точек или кавычек внутри названия).
4. Если анализ указывает на конкретный спорт, выбирай его.
"""
)

prefill = "```json\n"

# Предсказание модели для рассуждения и обработка данных

In [ ]:
thinking_model_system_prompt = (
    thinking_model_role_prompt
    + thinking_model_rule_prompt
    + thinking_prompt
    + thinking_model_few_shot_prompt
)
thinking_model_answers = process_dataframe_with_cache(
    test_df,
    thinking_model_system_prompt,
    thinking_model_prefill,
    "answer_thinking_model.json",
)
post_test_df = test_df.copy()
post_test_df = pd.concat(
    [
        test_df,
        pd.DataFrame(
            [answer for answer in thinking_model_answers], columns=["thinking"]
        ),
    ],
    axis=1,
)
post_test_df["title"] = post_test_df["title"].apply(
    lambda x: f"Входные данные:\n{x}\n"
) + post_test_df["thinking"].apply(lambda x: f"{thinking_model_prefill}{x}")
del post_test_df["thinking"]
post_test_df

  0%|          | 0/121 [00:00<?, ?it/s]

100%|██████████| 121/121 [17:35<00:00,  8.73s/it]


,title,item_id
0,"Входные данные:\nНабор натуральной мышечной массы\nАнализ: Набор натуральной мышечной массы — это термин, используемый в спортивном питании для обозначения программы питания и тренировок, направленных на прирост мышечной массы без использования допинга или синтетических добавок. Категория: Спортивное питание.",0
1,"Входные данные:\nФутзалки lunar gato\nАнализ: Футзалки — это специальные ботинки для игры в футзал, что является видом спорта, сочетающим элементы футбола и баскетбола. Модель Lunar Gato — это конкретная модель футзальных ботинок. Категория: Игры с мячом.",1
2,"Входные данные:\nМаты татами\nАнализ: Татами — это маты, на которые укладывают полы в бортах для занятийbudō, борьбы, йоги и других видов спорта. Категория: Фитнес и тренажёры.",2
3,"Входные данные:\nЮнион 27” abec 7 - полный аналог Penny Nikel\nАнализ: 27"" — размер диска, ABEC 7 — класс точности подшипника. Упоминание ""Юнион"" указывает на производителя ролевых колес. ""Полный аналог Penny Nikel"" означает, что это компонент для роликов или скейтборда. Категория: Ролики и скейтбординг.",3
4,"Входные данные:\nТермолосины для фигурного катания Mondor\nАнализ: Термолосины — это вид спортивной одежды, предназначенной для защиты от холода и сохранения тепла. Фигурное катание — это вид спорта, который включает в себя исполнение различных элементов на льду. Бренд Mondor специализируется на спортивной одежде.\n\nКатегория: Зимние виды спорта.",4
...,...,...
116,"Входные данные:\nEnergy diet банки\nАнализ: Energy diet банки — это вероятно емкости для пищевых продуктов, предназначенных для использования в спортивных целях, таких как снижение веса. Категория: Спортивное питание.",116
117,"Входные данные:\nКобура оперативная для пм\nАнализ: Кобура — это аксессуар для хранения и ношения оружия. Слова ""оперативная"" и упоминание ""пм"" (вероятно, означает ""пистолет Макарова"") указывают на то, что это устройство предназначено для ношения пистолета. В контексте маркетплейса, если упоминаются бренды типа Gletcher, Remington, Strike или слова ""кобура"", это относится к 'Пейнтбол и страйк...",117
118,Входные данные:\nГетры футбольные новые подросток\nАнализ: Гетры — это защитное equipment для ног при играх в футбол. Категория: Игры с мячом.,118
119,"Входные данные:\nДрайн гринвей\nАнализ: Драйн — это устройство для удаления воды с поля для гольфа. Гринвей относится к типу газонокосилок, но в данном контексте оно используется для ухода за газонами на полях для гольфа. Категория: Туризм.",119


# Решение

In [ ]:
system_prompt = role_prompt + rules_prompt  # + few_shot_prompt

In [ ]:
answers = process_dataframe_with_cache(post_test_df, system_prompt, prefill)
predicted_categories = []
for answer in answers:
    try:
        # Очищаем ответ markdown
        cleaned_answer = clean_json_response(answer)
        parsed_answer = json.loads(cleaned_answer)
        predicted_category = parsed_answer["category"]
    except JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        print(f"Ответ: {answer[:100]}...")
        predicted_category = "other"
    predicted_categories.append(predicted_category)

  0%|          | 0/121 [00:00<?, ?it/s]

100%|██████████| 121/121 [03:28<00:00,  1.72s/it]


In [ ]:
test_df_gpt = test_df.copy()
test_df_gpt["category"] = predicted_categories
test_df_gpt[["item_id", "category"]].to_csv("test_df_to_upload.csv", index=False)

In [47]:
test_df_gpt

,title,item_id,category
0,Набор натуральной мышечной массы,0,Спортивное питание
1,Футзалки lunar gato,1,Игры с мячом
2,Маты татами,2,Фитнес и тренажёры
3,Юнион 27” abec 7 - полный аналог Penny Nikel,3,Ролики и скейтбординг
4,Термолосины для фигурного катания Mondor,4,Зимние виды спорта
...,...,...,...
116,Energy diet банки,116,Спортивное питание
117,Кобура оперативная для пм,117,Пейнтбол и страйкбол
118,Гетры футбольные новые подросток,118,Игры с мячом
119,Драйн гринвей,119,Туризм
